In [ ]:
# Install required libraries
!pip install wfdb torch numpy scikit-learn tqdm

In [ ]:
import os
import sys
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import wfdb
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm  # Use notebook version of tqdm


## Configuration

In [ ]:

import os
import torch

class Config:
    # Project Paths
    BASE_DIR = os.getcwd()
    DATA_DIR = os.path.join(BASE_DIR, '..', 'data')
    MODELS_DIR = os.path.join(BASE_DIR, '..', 'models_saved')
    
    # Data Processing
    SAMPLING_RATE = 360
    WINDOW_SIZE = 180  # 90 before, 90 after R-peak
    INPUT_DIM = 1
    NUM_CLASSES = 5
    AAMI_CLASSES = ['N', 'S', 'V', 'F', 'Q']
    
    # Model Hyperparameters
    HIDDEN_DIM = 48
    NUM_LNN_LAYERS = 2
    ODE_STEPS = 6
    NUM_ATTENTION_HEADS = 4
    TAU_MIN = 0.1
    TAU_MAX = 10.0
    DROPOUT = 0.1
    
    # Training Hyperparameters
    BATCH_SIZE = 64
    LEARNING_RATE = 15e-4
    WEIGHT_DECAY = 1e-4
    NUM_EPOCHS = 1  # Arbitrary default, adjustable
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Clinical Thresholds (for Decision Logic)
    THETA_NORMAL = 0.9
    THETA_CRITICAL = 0.8  # Critical classes: 'V', 'F'
    
    @staticmethod
    def ensure_dirs():
        os.makedirs(Config.DATA_DIR, exist_ok=True)
        os.makedirs(Config.MODELS_DIR, exist_ok=True)


## LA-NN Models

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class LiquidTimeConstantCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, ode_steps, tau_min=0.1, tau_max=10.0):
        super(LiquidTimeConstantCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.ode_steps = ode_steps
        self.tau_min = tau_min
        self.tau_max = tau_max
        self.dt = 1.0 / ode_steps

        # Linear layers for Tau computation
        self.linear_tau_input = nn.Linear(input_dim, hidden_dim)
        self.linear_tau_hidden = nn.Linear(hidden_dim, hidden_dim)
        
        # Linear layers for f(.) - the nonlinearity
        self.linear_f_input = nn.Linear(input_dim, hidden_dim)
        self.linear_f_hidden = nn.Linear(hidden_dim, hidden_dim)

    def compute_tau(self, x, h):
        # Ï„ = Ï„min + (Ï„max - Ï„min) * Ïƒ(Linear(Input) + Linear(Hidden) + Bias)
        gate = self.linear_tau_input(x) + self.linear_tau_hidden(h)
        sig = torch.sigmoid(gate)
        tau = self.tau_min + (self.tau_max - self.tau_min) * sig
        return tau

    def compute_f(self, x, h):
        # f(.) typically involves a tanh or sigmoid activation
        # The thesis just says f(.), assuming standard RNN-like nonlinearity
        return torch.tanh(self.linear_f_input(x) + self.linear_f_hidden(h))

    def ode_step(self, x, h, tau, f_val):
        # dh/dt = (-h + f(.)) / tau
        # Euler integration: h(t+dt) = h(t) + dh/dt * dt
        dh_dt = (-h + f_val) / tau
        h_new = h + dh_dt * self.dt
        return h_new

    def forward(self, x, h_prev=None):
        if h_prev is None:
            h_prev = torch.zeros(x.size(0), self.hidden_dim).to(x.device)
        
        h = h_prev
        
        # We assume x is constant over the ODE steps for this time step, 
        # or we integrate the dynamics. 
        # For standard LNN/LTC, we compute params based on current input and state, 
        # then evolve the state.
        
        for _ in range(self.ode_steps):
            # Recalculate Tau and f at each micro-step? 
            # Usually LTC keeps input constant but state evolves.
            # State `h` is evolving.
            
            tau = self.compute_tau(x, h)
            f_val = self.compute_f(x, h)
            h = self.ode_step(x, h, tau, f_val)
            
        return h

class LNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, ode_steps, tau_min=0.1, tau_max=10.0):
        super(LNNEncoder, self).__init__()
        self.num_layers = num_layers
        self.layers = nn.ModuleList()
        
        for i in range(num_layers):
            # First layer takes input_dim, subsequent layers take hidden_dim
            in_d = input_dim if i == 0 else hidden_dim
            self.layers.append(
                LiquidTimeConstantCell(in_d, hidden_dim, ode_steps, tau_min, tau_max)
            )

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size, seq_len, _ = x.size()
        
        # Initialize hidden states for each layer
        hidden_states = [None] * self.num_layers
        
        outputs = []
        
        for t in range(seq_len):
            x_t = x[:, t, :]
            
            for layer_idx, layer in enumerate(self.layers):
                h_prev = hidden_states[layer_idx]
                h_new = layer(x_t, h_prev)
                hidden_states[layer_idx] = h_new
                x_t = h_new # Feed output of this layer as input to next
            
            outputs.append(x_t.unsqueeze(1))
            
        # Concatenate outputs along time dimension
        # Shape: (batch_size, seq_len, hidden_dim)
        return torch.cat(outputs, dim=1)


In [ ]:

import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0) # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttentionBlock, self).__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
        self.positional_encoding = PositionalEncoding(d_model)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        
        # Add positional encoding
        x = self.positional_encoding(x)
        
        # Pre-Norm Attention
        # Residual connection: x + Attention(Norm(x))
        x_norm = self.ln1(x)
        attn_out, attn_weights = self.mha(x_norm, x_norm, x_norm)
        x = x + self.dropout(attn_out)
        
        # Pre-Norm FFN
        # Residual connection: x + FFN(Norm(x))
        x_norm = self.ln2(x)
        ffn_out = self.ffn(x_norm)
        x = x + ffn_out
        
        return x, attn_weights


In [ ]:

import torch
import torch.nn as nn

class LANN(nn.Module):
    def __init__(self, config):
        super(LANN, self).__init__()
        
        self.input_dim = config.INPUT_DIM
        self.hidden_dim = config.HIDDEN_DIM
        self.num_lnn_layers = config.NUM_LNN_LAYERS
        self.ode_steps = config.ODE_STEPS
        self.tau_min = config.TAU_MIN
        self.tau_max = config.TAU_MAX
        self.num_heads = config.NUM_ATTENTION_HEADS
        self.num_classes = config.NUM_CLASSES
        self.dropout_rate = config.DROPOUT
        
        # 1. LNN Encoder
        self.lnn_encoder = LNNEncoder(
            input_dim=self.input_dim,
            hidden_dim=self.hidden_dim,
            num_layers=self.num_lnn_layers,
            ode_steps=self.ode_steps,
            tau_min=self.tau_min,
            tau_max=self.tau_max
        )
        
        # 2. Multi-Head Attention Block
        self.attention_block = MultiHeadAttentionBlock(
            d_model=self.hidden_dim,
            num_heads=self.num_heads,
            dropout=self.dropout_rate
        )
        
        # 3. Classification Head
        # "The final aggregated latent vector is passed through a classification layer"
        self.classifier = nn.Sequential(
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dim, self.num_classes)
        )

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        
        # LNN Encoder
        lnn_out = self.lnn_encoder(x) # (batch_size, seq_len, hidden_dim)
        
        # Attention Block
        attn_out, attn_weights = self.attention_block(lnn_out) # (batch_size, seq_len, hidden_dim)
        
        # Global Average Pooling (Aggregation)
        # Check if we should use GAP or specific token. Using GAP is standard for sequence classification without CLS token.
        aggregated_features = torch.mean(attn_out, dim=1) # (batch_size, hidden_dim)
        
        # Classification
        logits = self.classifier(aggregated_features) # (batch_size, num_classes)
        
        return logits, attn_weights

    def get_lnn_taus(self, x):
        """
        Helper method to extract Tau values for interpretability analysis.
        This would require modifying LNNEncoder to return taus.
        Currently a placeholder.
        """
        pass


## Data Pipeline

In [ ]:
import os
import numpy as np
import torch
import wfdb
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Mapping from MIT-BIH annotations to AAMI classes
# Class indices: N=0, S=1, V=2, F=3, Q=4
AAMI_MAPPING = {
    'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,  # Normal
    'A': 1, 'a': 1, 'J': 1, 'S': 1,          # Supraventricular (S)
    'V': 2, 'E': 2,                          # Ventricular (V)
    'F': 3,                                  # Fusion (F)
    '/': 4, 'f': 4, 'Q': 4                   # Unknown/Paced (Q)
    # Ignored: [, ], !, x, (, ), p, t, u, `, ', ~, +, "
}

class ECGDataset(Dataset):
    def __init__(self, signals, labels):
        """
        Args:
            signals (np.array): Shape (num_samples, seq_len, input_dim)
            labels (np.array): Shape (num_samples,)
        """
        self.signals = torch.FloatTensor(signals)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

def generate_mock_data(num_samples=1000, seq_len=180, input_dim=1, num_classes=5):
    """
    Generates random mock data for testing the pipeline.
    """
    signals = np.random.randn(num_samples, seq_len, input_dim).astype(np.float32)
    labels = np.random.randint(0, num_classes, size=(num_samples,))
    return signals, labels

def normalize_signal(signal):
    """Z-score normalization."""
    mean = np.mean(signal)
    std = np.std(signal)
    if std == 0:
        return signal - mean
    return (signal - mean) / std

def get_records(db_dir):
    """
    Returns a list of record names from the MIT-BIH Arrhythmia Database.
    Excludes records with significant noise or paced rhythms if necessary, 
    but for now we include the standard set.
    """
    # Standard MIT-BIH records
    records = [
        '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '111', 
        '112', '113', '114', '115', '116', '117', '118', '119', '121', '122', '123', '124',
        '200', '201', '202', '203', '205', '207', '208', '209', '210', '212', '213', 
        '214', '215', '219', '220', '221', '222', '223', '228', '230', '231', '232', '233', '234'
    ]
    return records

def process_record(record_name, db_dir, window_size):
    """
    Reads a WFDB record, extracts beats based on annotations, and labels them.
    """
    # Read signal
    record = wfdb.rdrecord(os.path.join(db_dir, record_name))
    signal = record.p_signal[:, 0] # Use lead MLII (usually channel 0)
    
    # Read annotations
    annotation = wfdb.rdann(os.path.join(db_dir, record_name), 'atr')
    symbols = annotation.symbol
    samples = annotation.sample
    
    processed_beats = []
    processed_labels = []
    
    half_window = window_size // 2
    
    for i, symbol in enumerate(symbols):
        if symbol in AAMI_MAPPING:
            r_peak = samples[i]
            
            # Check boundaries
            if r_peak - half_window < 0 or r_peak + half_window > len(signal):
                continue
                
            # Segment
            beat = signal[r_peak - half_window : r_peak + half_window]
            
            # Normalize
            beat = normalize_signal(beat)
            
            processed_beats.append(beat)
            processed_labels.append(AAMI_MAPPING[symbol])
            
    return processed_beats, processed_labels

def load_data(config):
    """
    Main function to load and preprocess data from MIT-BIH.
    """
    print(f"Loading Data from {config.DATA_DIR}...")
    
    db_name = 'mitdb'
    db_dir = os.path.join(config.DATA_DIR, db_name)
    
    # 1. Check if database exists, if not download
    if not os.path.exists(db_dir):
        print(f"Downloading {db_name} to {db_dir}...")
        os.makedirs(db_dir, exist_ok=True)
        wfdb.dl_database(db_name, db_dir)
        print("Download complete.")
    else:
        print(f"Database found at {db_dir}")

    # 2. Process Records
    all_signals = []
    all_labels = []
    
    records = get_records(db_dir)
    print(f"Processing {len(records)} records...")
    
    for rec in tqdm(records):
        try:
            beats, labels = process_record(rec, db_dir, config.WINDOW_SIZE)
            all_signals.extend(beats)
            all_labels.extend(labels)
        except Exception as e:
            print(f"Error processing record {rec}: {e}")
            
    # Convert to numpy
    X = np.array(all_signals)
    y = np.array(all_labels)
    
    # Reshape X to (num_samples, seq_len, input_dim)
    # The current shape is (num_samples, seq_len). 
    # We need to add the feature dimension.
    X = X[..., np.newaxis]
    
    print(f"Total Heartbeats Processed: {X.shape[0]}")
    print(f"Signal Shape: {X.shape}, Label Shape: {y.shape}")
    
    # 3. Stratified Split (Train: 70%, Val: 15%, Test: 15%)
    # First split: Train (70%) vs Temp (30%)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=42
    )
    
    # Second split: Val (15% of total -> 50% of Temp) vs Test (15% of total -> 50% of Temp)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )
    
    print(f"Train/Val/Test Splits: {len(X_train)}/{len(X_val)}/{len(X_test)}")
    
    # 4. Create Datasets and Loaders
    train_dataset = ECGDataset(X_train, y_train)
    val_dataset = ECGDataset(X_val, y_val)
    test_dataset = ECGDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)
    
    return train_loader, val_loader, test_loader


## Trainer

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import time


class Trainer:
    def __init__(self, model, config, train_loader, val_loader):
        self.model = model
        self.config = config
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = config.DEVICE
        
        # Balanced Class Weights (Approximate inverse frequency)
        # N: 1.0 (Majority)
        # S: ~30.0 
        # V: ~15.0
        # F: ~100.0 (Minority)
        # Q: ~15.0
        class_weights = torch.tensor([1.0, 30.0, 15.0, 100.0, 15.0]).to(self.device)
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        self.optimizer = optim.AdamW(
            self.model.parameters(), 
            lr=config.LEARNING_RATE, 
            weight_decay=config.WEIGHT_DECAY
        )
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, 
            T_0=10, 
            T_mult=2
        )
        
        self.model.to(self.device)

    def train_epoch(self, epoch):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config.NUM_EPOCHS} [Train]")
        for signals, labels in pbar:
            signals, labels = signals.to(self.device), labels.to(self.device)
            
            self.optimizer.zero_grad()
            outputs, _ = self.model(signals)
            loss = self.criterion(outputs, labels)
            loss.backward()
            
            # Gradient Clipping to prevent explosion (common in LNNs)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            self.optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({'loss': running_loss/len(self.train_loader), 'acc': 100 * correct / total})
            
        return running_loss / len(self.train_loader), 100 * correct / total

    def validate(self, epoch):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for signals, labels in self.val_loader:
                signals, labels = signals.to(self.device), labels.to(self.device)
                outputs, _ = self.model(signals)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        acc = 100 * correct / total
        loss = running_loss / len(self.val_loader)
        print(f"Epoch {epoch+1} [Val]: Loss: {loss:.4f}, Acc: {acc:.2f}%")
        return loss, acc

    def train(self):
        print(f"Starting training on {self.device}...")
        best_acc = 0.0
        
        for epoch in range(self.config.NUM_EPOCHS):
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.validate(epoch)
            
            self.scheduler.step()
            
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(self.model.state_dict(), f"{self.config.MODELS_DIR}/la_nn_best.pth")
                print("Saved new best model.")
                
        print("Training complete.")


## Evaluation Metrics

In [ ]:

import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch.nn.functional as F

def evaluate_model(model, loader, config):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    device = config.DEVICE
    
    with torch.no_grad():
        for signals, labels in loader:
            signals = signals.to(device)
            outputs, _ = model(signals)
            probs = F.softmax(outputs, dim=1)
            
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
            
    # Classification Report
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=config.AAMI_CLASSES))
    
    # Clinical/Decision Logic Simulation
    apply_decision_logic(all_probs, all_labels, config)

def apply_decision_logic(probs, true_labels, config):
    """
    Applies the tiered alert system described in the thesis.
    Level 1: Normal (N > theta_normal)
    Level 3: Critical (V or F > theta_critical)
    Level 2: Monitor (Everything else)
    """
    print("\n--- Clinical Decision Logic Simulation ---")
    
    probs = np.array(probs)
    true_labels = np.array(true_labels)
    
    # Class indices
    # ['N', 'S', 'V', 'F', 'Q'] -> [0, 1, 2, 3, 4]
    idx_N = 0
    idx_V = 2
    idx_F = 3
    
    total = len(true_labels)
    alarms_triggered = 0
    correct_alarms = 0
    missed_alarms = 0 # Critical events not alarmed
    
    for i in range(total):
        prob_vector = probs[i]
        true_label = true_labels[i]
        
        # Decision Logic
        action = "MONITOR" # Default Level 2
        
        if prob_vector[idx_N] > config.THETA_NORMAL:
            action = "NORMAL"
        elif (prob_vector[idx_V] > config.THETA_CRITICAL) or (prob_vector[idx_F] > config.THETA_CRITICAL):
            action = "ALARM"
        
        # Validation
        if action == "ALARM":
            alarms_triggered += 1
            if true_label in [idx_V, idx_F]:
                correct_alarms += 1
        
        # Check for missed critical events (False Negatives for Alarm)
        if true_label in [idx_V, idx_F] and action != "ALARM":
            missed_alarms += 1
            
    print(f"Total Samples: {total}")
    print(f"Total Alarms Triggered: {alarms_triggered}")
    print(f"True Alarms (Correctly identified V/F): {correct_alarms}")
    print(f"False Alarms: {alarms_triggered - correct_alarms}")
    print(f"Missed Critical Events (Critical FN): {missed_alarms}")


## Main Execution

In [ ]:
def main():
    print("Initializing LA-NN Project on Colab...")
    
    # 1. Setup
    Config.ensure_dirs()
    print(f"Device: {Config.DEVICE}")
    
    # 2. Data Loading
    # Force re-download check since Colab is ephemeral
    train_loader, val_loader, test_loader = load_data(Config)
    
    # 3. Model Initialization
    model = LANN(Config)
    print("Model Architecture:")
    # print(model) # Optional: comment out to save space
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total Trainable Parameters: {total_params}")
    
    # 4. Training
    trainer = Trainer(model, Config, train_loader, val_loader)
    trainer.train()
    
    # 5. Evaluation (Load best model)
    print("\nLoading best model for evaluation...")
    if os.path.exists(f"{Config.MODELS_DIR}/la_nn_best.pth"):
        model.load_state_dict(torch.load(f"{Config.MODELS_DIR}/la_nn_best.pth", map_location=Config.DEVICE))
    else:
        print("Warning: Model file not found (maybe training didn't finish?), using current weights.")
        
    evaluate_model(model, test_loader, Config)

if __name__ == "__main__":
    main()